In [2]:
import yfinance as yf
import pandas as pd

# Download 2 years of Apple data
df = yf.download("AAPL", period="2y")

# Flatten multi-level columns if they appear (yfinance sometimes does this)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(df.shape)   # (rows, columns)
print(df.head())


[*********************100%***********************]  1 of 1 completed

(501, 5)
Price            Close        High         Low        Open    Volume
Date                                                                
2024-06-25  207.274490  209.564649  206.818434  207.353790  55549700
2024-06-26  211.418610  213.014784  208.831024  209.683639  66213200
2024-06-27  212.261307  213.887222  210.526336  212.846236  49772700
2024-06-28  208.811157  214.214364  208.493913  213.916937  82542700
2024-07-01  214.888519  215.641987  210.099998  210.268536  60402900


In [3]:
# Select one column → gives you a Series (a single list with an index)
close_prices = df["Close"]
print(type(close_prices))   # <class 'pandas.core.series.Series'>
print(close_prices.head())

# Select multiple columns → gives you a smaller DataFrame
ohlc = df[["Open", "High", "Low", "Close"]]
print(ohlc.head())


<class 'pandas.core.series.Series'>
Date
2024-06-25    207.274490
2024-06-26    211.418610
2024-06-27    212.261307
2024-06-28    208.811157
2024-07-01    214.888519
Name: Close, dtype: float64
Price             Open        High         Low       Close
Date                                                      
2024-06-25  207.353790  209.564649  206.818434  207.274490
2024-06-26  209.683639  213.014784  208.831024  211.418610
2024-06-27  212.846236  213.887222  210.526336  212.261307
2024-06-28  213.916937  214.214364  208.493913  208.811157
2024-07-01  210.268536  215.641987  210.099998  214.888519


In [4]:
# You can do math on entire columns at once — no for loops needed

# How much did the price move each day? (High - Low)
df["Daily_Range"] = df["High"] - df["Low"]

# What was the % change from open to close?
df["OC_Change"] = (df["Close"] - df["Open"]) / df["Open"] * 100

print(df[["Open", "Close", "Daily_Range", "OC_Change"]].head())


Price             Open       Close  Daily_Range  OC_Change
Date                                                      
2024-06-25  207.353790  207.274490     2.746215  -0.038244
2024-06-26  209.683639  211.418610     4.183760   0.827423
2024-06-27  212.846236  212.261307     3.360886  -0.274813
2024-06-28  213.916937  208.811157     5.720451  -2.386805
2024-07-01  210.268536  214.888519     5.541989   2.197182


In [5]:
# Show only days where price went up (Close > Open)
up_days = df[df["Close"] > df["Open"]]
print(f"Up days: {len(up_days)} out of {len(df)}")

# Show only days where volume was unusually high
high_volume = df[df["Volume"] > df["Volume"].mean() * 1.5]
print(f"High volume days: {len(high_volume)}")
print(high_volume[["Close", "Volume"]].head())


Up days: 273 out of 501
High volume days: 43
Price            Close     Volume
Date                             
2024-06-28  208.811157   82542700
2024-08-02  217.971832  105568600
2024-08-05  207.472763  119548600
2024-09-20  226.502075  318679900
2024-11-25  231.391617   90152800


In [6]:
print(df.describe())                                          # summary stats
print(df.dtypes)                                              # column types
print(df.isnull().sum())                                      # missing values
print(df.sort_values("Volume", ascending=False).head())       # sort by column
print(df.index)                                               # check date index


Price       Close        High         Low        Open        Volume  \
count  501.000000  501.000000  501.000000  501.000000  5.010000e+02   
mean   240.096866  242.517759  237.506376  239.890675  5.154607e+07   
std     28.395296   28.557405   28.425236   28.508722  2.343373e+07   
min    171.513763  189.339575  168.320644  171.046232  1.791060e+07   
25%    219.679260  222.283679  217.380448  219.538272  3.944620e+07   
50%    234.093109  235.714538  231.088216  233.319312  4.620830e+07   
75%    260.250214  261.988631  257.282109  259.490937  5.520210e+07   
max    315.200012  317.399994  309.649994  314.179993  3.186799e+08   

Price  Daily_Range   OC_Change  
count   501.000000  501.000000  
mean      5.011383    0.104196  
std       2.707996    1.517993  
min       1.707204   -7.648634  
25%       3.363746   -0.592838  
50%       4.395944    0.104011  
75%       5.859042    0.782000  
max      28.569050   15.644088  
Price
Close          float64
High           float64
Low        

> 🔍 **`describe()` is your best friend.** Run it on every new dataset. It shows you min, max, mean, and std for every column in one line.


In [7]:
# Always do this first on any new dataset
print(df.isnull().sum())

# See which rows specifically have missing values
print(df[df.isnull().any(axis=1)])


Price
Close          0
High           0
Low            0
Open           0
Volume         0
Daily_Range    0
OC_Change      0
dtype: int64
Empty DataFrame
Columns: [Close, High, Low, Open, Volume, Daily_Range, OC_Change]
Index: []


In [8]:
# Strategy 1: Drop rows with missing values
# Use when: very few rows are missing (< 1% of data)
df_dropped = df.dropna()
print(f"Rows before: {len(df)} | After dropping NaN: {len(df_dropped)}")

# Strategy 2: Forward fill — carry the last known value forward
# Use when: time series data where you want continuity
df_ffill = df.ffill()
# If Monday's close is missing, it gets filled with Friday's close

# Strategy 3: Fill with a specific value
# Use when: volume is missing (assume 0 = no trading that day)
df["Volume"] = df["Volume"].fillna(0)


Rows before: 501 | After dropping NaN: 501


In [9]:
# for expmple if you want to drop the "Dividends" and "Stock Splits" columns if they exist, you can do:
# for our dataset we dont have these columns but this is how you would do it in general
cols_to_drop = [c for c in ["Dividends", "Stock Splits"] if c in df.columns]
df = df.drop(columns=cols_to_drop)
print("Cleaned columns:", df.columns.tolist())


Cleaned columns: ['Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Range', 'OC_Change']


In [10]:
print(f"Duplicate rows: {df.index.duplicated().sum()}")
df = df[~df.index.duplicated(keep="last")]


Duplicate rows: 0


In [11]:
import os
os.makedirs("data/processed", exist_ok=True)

df.to_csv("data/processed/AAPL_clean.csv")
print("Saved!")

# Load it back any time without re-downloading
df = pd.read_csv("data/processed/AAPL_clean.csv", index_col="Date", parse_dates=True)


Saved!


In [12]:
# Run this once in terminal to install
# pip install matplotlib seaborn


In [13]:
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(df.index, df["Close"], linewidth=1.5, color="#185FA5")
plt.title("Apple (AAPL) — Closing Price", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Price ")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(df.index, df['Close'], color='#185FA5', linewidth=1.2)
axes[0].set_title('AAPL — Closing Price', fontsize=13)
axes[0].set_ylabel('Price (USD)')

axes[1].bar(df.index, df['Volume'], color='#5DCAA5', alpha=0.6, width=1)
axes[1].set_title('Daily Volume', fontsize=13)
axes[1].set_ylabel('Shares Traded')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.savefig("./plots/price_and_volume.png", dpi=150)
plt.show()


In [ ]:
tickers = ["AAPL", "TSLA", "GOOGL"]
raw = yf.download(tickers, period="1y")["Close"]


# Normalize: divide every column by its first value, multiply by 100
# This makes all stocks start at 100 so you compare performance, not price
normalized = raw / raw.iloc[0] * 100

plt.figure(figsize=(13, 5))
for col in normalized.columns:
    plt.plot(normalized.index, normalized[col], label=col, linewidth=1.5)

plt.axhline(100, color='black', linestyle='--', linewidth=0.7, alpha=0.5)
plt.title('Normalized Performance — Base 100 at Start', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Normalized Price')
plt.legend()
plt.tight_layout()
plt.savefig("./plots/multi_stock.png", dpi=150)
plt.show()


In [ ]:
df["MA20"] = df["Close"].rolling(window=20).mean()
df["MA50"] = df["Close"].rolling(window=50).mean()

plt.figure(figsize=(13, 5))
plt.plot(df.index, df['Close'], label='Close Price', alpha=0.6, linewidth=1)
plt.plot(df.index, df['MA20'], label='20-Day MA', linewidth=2, color='orange')
plt.plot(df.index, df['MA50'], label='50-Day MA', linewidth=2, color='red')
plt.title('AAPL — Price with Moving Averages', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig("./plots/moving_averages.png", dpi=150)
plt.show()


In [ ]:
df["Daily_Return"] = df["Close"].pct_change()

plt.figure(figsize=(13, 4))
plt.plot(df.index, df['Daily_Return'], linewidth=0.8, color='#E05252', alpha=0.8)
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('AAPL — Daily Returns (%)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Daily Return')
plt.tight_layout()
plt.savefig("./plots/daily_returns.png", dpi=150)
plt.show()


In [ ]:
print(df.describe())


In [ ]:
returns = df['Daily_Return'].dropna()   
print(f'Average daily return : {returns.mean()*100}%')
# - **Average daily return** → Average percentage change in stock price per day  

print(f'Std deviation (risk) : {returns.std()*100}%')
# - **Std deviation (risk)** → Measures how volatile the stock is. Higher value means higher risk and bigger price swings.  

print(f'Best single day      : {returns.max()*100}%')
# - **Best single day** → Highest one-day gain in the dataset  

print(f'Worst single day     : {returns.min()*100}%')
# - **Worst single day** → Largest one-day loss in the dataset  

In [ ]:
df['Daily_Return'].hist(bins=50)
plt.title('Distribution of Daily Returns')
plt.xlabel('Daily Return')
plt.ylabel('Frequency')

plt.show()


In [ ]:
cols = ["Open", "High", "Low", "Close", "Volume", "Daily_Return", "MA20"]
corr = df[cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig("./plots/correlation_heatmap.png", dpi=150)
plt.show()


**Write your observation below:**


In [ ]:
df["Day_of_Week"] = df.index.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
dow_avg = df.groupby('Day_of_Week')['Daily_Return'].mean().reindex(order)

plt.figure(figsize=(8, 4))
dow_avg.plot(kind='bar', color='#185FA5', alpha=0.8)
plt.axhline(0, color='black', linestyle='--', linewidth=0.7)
plt.title('Average Daily Return by Day of Week', fontsize=14)
plt.xlabel('Day')
plt.ylabel('Avg Return')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("./plots/day_of_week.png", dpi=150)
plt.show()
